In [ ]:
%load_ext autoreload
%autoreload 2

import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from vetuner.correction import gated_correction
from vetuner.drive_cycle import VALIDATION_CYCLE, generate_drive_cycle
from vetuner.engine_model import simulate_engine
from vetuner.filtering import (FILTER_METHODS,filter_log,kalman_steady_state,measure_phase_lag,)
from vetuner.metrics import evaluate_afr, evaluate_table
from vetuner.sensors import NOISE_FREE, SensorConfig, apply_sensor_model
from vetuner.ve_surface import BASE_MAP_SURFACE, TRUE_SURFACE, ve_table

train = generate_drive_cycle()
test = generate_drive_cycle(cycle=VALIDATION_CYCLE)
base, truth = ve_table(params=BASE_MAP_SURFACE), ve_table(params=TRUE_SURFACE)

sim = simulate_engine(train, base)
log = apply_sensor_model(sim, seed=0)
clean_log = apply_sensor_model(sim, config=NOISE_FREE)

In [ ]:
"""
Compare three filters applied to the air-fuel ratio channel.
"""

settings = {
    "none": {},
    "moving_average": {"window": 11},
    "savitzky_golay": {"window": 11, "polyorder": 2},
    "kalman": {"process_variance": 2e-3, "measurement_variance": 1e-2},
}

print(f"{'filter':18s} {'noise sd':>9} {'lag':>5} {'RMSE':>7} {'visited':>8} "
      f"{'AFR%':>7} {'cover':>6}")

for method in FILTER_METHODS:
    filtered = filter_log(log, method, **settings[method])
    residual = float(np.std(filtered.afr - clean_log.afr))
    lag = measure_phase_lag(clean_log.afr, filtered.afr)

    result, gate = gated_correction(filtered, base)
    t = evaluate_table(result.table, truth, result.counts)
    a = evaluate_afr(result.table, test)

    print(f"{method:18s} {residual:9.4f} {lag:5d} {t.rmse:7.2f} {t.rmse_visited:8.2f} "
          f"{a.mean_abs_pct:7.2f} {t.coverage:6.0%}")

state = kalman_steady_state(2e-3, 1e-2)
print(f"\nKalman steady-state gain {state.gain:.3f}, "
      f"equivalent time constant {state.equivalent_tau_samples:.1f} samples "
      f"({state.equivalent_tau_samples * 0.02:.3f} s)")

In [ ]:
"""
Sweep the Savitzky-Golay window to find where over-filtering starts.
"""

print(f"{'window':>7} {'noise sd':>9} {'RMSE':>7} {'visited':>8} {'AFR%':>7}")
for window in (5, 11, 21, 51, 101, 201):
    filtered = filter_log(log, "savitzky_golay", window=window, polyorder=2)
    residual = float(np.std(filtered.afr - clean_log.afr))
    result, _ = gated_correction(filtered, base)
    t = evaluate_table(result.table, truth, result.counts)
    a = evaluate_afr(result.table, test)
    print(f"{window:7d} {residual:9.4f} {t.rmse:7.2f} {t.rmse_visited:8.2f} "
          f"{a.mean_abs_pct:7.2f}")

In [ ]:
"""
Check what the residual column is comparing against. 
The lag-free reference is the clean log, while the lag-matched reference is the log with only lag applied.
"""

lag_only = apply_sensor_model(sim, config=SensorConfig(afr_noise_std=0.0, afr_quantisation=0.0))
print(f"residual vs lag-free reference:  {np.std(log.afr - clean_log.afr):.4f}")
print(f"residual vs lag-matched reference: {np.std(log.afr - lag_only.afr):.4f}")